# Taller Métricas y Datos 2
## Haider Johan Arango Diaz - 160004613

---
## Actividad 1: Lectura de datasets, creación de campos y métricas de distancia

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform, cosine, euclidean, hamming, jaccard
from scipy.stats import pearsonr
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Lectura de los datasets
df_demanda = pd.read_excel('demanda_laboral.xlsx')
df_encuesta = pd.read_excel('encuesta.xls')

print('=== Dataset demanda_laboral ===')
print(f'Dimensiones: {df_demanda.shape}')
display(df_demanda.head())

print('\n=== Dataset encuesta ===')
print(f'Dimensiones: {df_encuesta.shape}')
display(df_encuesta.head())

### 1a. Creación del campo SMLMV en demanda_laboral

Se crea el campo **SMLMV** dividiendo el campo `Ingresos` entre el valor del Salario Mínimo Legal Mensual Vigente (SMLMV) de **$1,750,905 COP**.

**Justificación:** Se define esta constante (`SMLMV_VALOR`) de forma separada para facilitar su actualización cuando el salario mínimo cambie cada año. De esta forma, solo se necesita modificar un único valor en el código para recalcular toda la columna, lo que garantiza mantenibilidad y consistencia en el análisis.

In [ ]:
# Constante SMLMV - valor actualizable
# Al cambiar solo esta constante se actualiza toda la columna SMLMV
SMLMV_VALOR = 1750905

df_demanda['SMLMV'] = df_demanda['Ingresos'] / SMLMV_VALOR

print('Campo SMLMV creado exitosamente.')
print(f'Constante SMLMV utilizada: ${SMLMV_VALOR:,.0f} COP')
print(f'\nEjemplo - Ingresos vs SMLMV:')
display(df_demanda[['Ingresos', 'SMLMV']].head(10))

### 1a. Creación del campo "I. Fuera Servicios" en encuesta

Se crea el campo **I. Fuera Servicios** con la fórmula:

```
I. Fuera Servicios = I. Mensual - G. Alimentación - (Consumo Energía × Precio kWh) - (Consumo de Agua × Precio m³)
```

**Valores de referencia (Villavicencio, Colombia - promedio 2015):**
- **Precio kWh:** ~$450 COP/kWh (tarifa residencial promedio según datos del sector eléctrico colombiano 2015)
- **Precio m³ agua:** ~$1,300 COP/m³ (tarifa residencial promedio de acueducto en Villavicencio, 2015)

Dado que `I. Mensual` y `G. Alimentación` están en **miles de pesos**, los precios de servicios se convierten a la misma unidad (dividiéndolos entre 1000).

**Justificación:** Este campo se crea para estimar mejor los **ingresos libres** del encuestado después de descontar los gastos esenciales (alimentación, energía y agua). Esto permite una mejor gestión y análisis de gastos en relación con las demás variables del dataset, ofreciendo una visión más realista de la capacidad económica disponible de cada individuo.

In [ ]:
# Constantes de precios de servicios públicos - Villavicencio 2015 (promedios)
# Estos valores pueden actualizarse al cambiar de año o región
PRECIO_KWH_COP = 450    # COP por kWh - tarifa residencial promedio
PRECIO_M3_AGUA_COP = 1300  # COP por m3 - tarifa residencial promedio

# Conversión a miles de pesos (misma unidad que I. Mensual y G. Alimentación)
PRECIO_KWH = PRECIO_KWH_COP / 1000
PRECIO_M3_AGUA = PRECIO_M3_AGUA_COP / 1000

df_encuesta['I. Fuera Servicios'] = (
    df_encuesta['I. Mensual']
    - df_encuesta['G. Alimentación']
    - (df_encuesta['Consumo Energía'] * PRECIO_KWH)
    - (df_encuesta['Consumo de Agua'] * PRECIO_M3_AGUA)
)

print('Campo "I. Fuera Servicios" creado exitosamente.')
print(f'Precio kWh utilizado: ${PRECIO_KWH_COP} COP = {PRECIO_KWH} (miles de pesos)')
print(f'Precio m³ agua utilizado: ${PRECIO_M3_AGUA_COP} COP = {PRECIO_M3_AGUA} (miles de pesos)')
print(f'\nEjemplo de cálculo:')
display(df_encuesta[['I. Mensual', 'G. Alimentación', 'Consumo Energía', 'Consumo de Agua', 'I. Fuera Servicios']].head(10))

### 1b. Matrices de distancia - Dataset demanda_laboral

Se calculan las métricas: **Similitud Coseno**, **Distancia Euclidiana**, **Hamming** y **Jaccard**.

Para el cálculo se utilizan las columnas numéricas del dataset. Los datos se normalizan para que las métricas sean comparables entre sí.

In [ ]:
# Seleccionar columnas numéricas y normalizar para demanda_laboral
cols_demanda = ['Ingresos', 'Edad', 'Tiempo', 'Posgrado', 'Sexo', 'Naturaleza', 'SMLMV']
df_dem_num = df_demanda[cols_demanda].dropna().copy()

# Normalización min-max para distancias
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_dem_norm = pd.DataFrame(scaler.fit_transform(df_dem_num), columns=cols_demanda)

# Binarización para Hamming y Jaccard (umbral en la mediana)
df_dem_bin = (df_dem_norm > df_dem_norm.median()).astype(int)

n = len(df_dem_norm)
print(f'Número de registros a comparar: {n}')
print(f'Total de pares: {n*(n-1)//2}')

In [ ]:
# Calcular matrices de distancia para demanda_laboral
print('Calculando matrices de distancia para demanda_laboral...')

# Distancia Coseno (1 - similitud coseno)
dist_coseno_dem = squareform(pdist(df_dem_norm.values, metric='cosine'))
mat_coseno_dem = pd.DataFrame(dist_coseno_dem)
print('✓ Matriz Coseno calculada')

# Distancia Euclidiana
dist_euclid_dem = squareform(pdist(df_dem_norm.values, metric='euclidean'))
mat_euclid_dem = pd.DataFrame(dist_euclid_dem)
print('✓ Matriz Euclidiana calculada')

# Distancia Hamming (sobre datos binarizados)
dist_hamming_dem = squareform(pdist(df_dem_bin.values, metric='hamming'))
mat_hamming_dem = pd.DataFrame(dist_hamming_dem)
print('✓ Matriz Hamming calculada')

# Distancia Jaccard (sobre datos binarizados)
dist_jaccard_dem = squareform(pdist(df_dem_bin.values, metric='jaccard'))
mat_jaccard_dem = pd.DataFrame(dist_jaccard_dem)
print('✓ Matriz Jaccard calculada')

# Mostrar fragmento de cada matriz
print('\n--- Matriz Coseno (primeros 5x5) ---')
display(mat_coseno_dem.iloc[:5, :5].round(4))

print('\n--- Matriz Euclidiana (primeros 5x5) ---')
display(mat_euclid_dem.iloc[:5, :5].round(4))

print('\n--- Matriz Hamming (primeros 5x5) ---')
display(mat_hamming_dem.iloc[:5, :5].round(4))

print('\n--- Matriz Jaccard (primeros 5x5) ---')
display(mat_jaccard_dem.iloc[:5, :5].round(4))

### 1c y 1d. Top 5 registros más similares y más disímiles - demanda_laboral

In [ ]:
def get_top_pairs(dist_matrix, n_top=5, most_similar=True):
    """Obtiene los top N pares más similares o disímiles de una matriz de distancia."""
    n = dist_matrix.shape[0]
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            pairs.append((i, j, dist_matrix[i, j]))
    pairs.sort(key=lambda x: x[2], reverse=not most_similar)
    return pairs[:n_top]

# Usando distancia Euclidiana como métrica principal
print('=== Top 5 registros MÁS SIMILARES (menor distancia Euclidiana) - demanda_laboral ===')
top_sim_dem = get_top_pairs(dist_euclid_dem, 5, most_similar=True)
for rank, (i, j, dist) in enumerate(top_sim_dem, 1):
    print(f'\n{rank}. Registros {i} y {j} - Distancia: {dist:.6f}')
    print(f'   Registro {i}: {df_dem_num.iloc[i].to_dict()}')
    print(f'   Registro {j}: {df_dem_num.iloc[j].to_dict()}')

print('\n\n=== Top 5 registros MÁS DISÍMILES (mayor distancia Euclidiana) - demanda_laboral ===')
top_dis_dem = get_top_pairs(dist_euclid_dem, 5, most_similar=False)
for rank, (i, j, dist) in enumerate(top_dis_dem, 1):
    print(f'\n{rank}. Registros {i} y {j} - Distancia: {dist:.6f}')
    print(f'   Registro {i}: {df_dem_num.iloc[i].to_dict()}')
    print(f'   Registro {j}: {df_dem_num.iloc[j].to_dict()}')

### 1b. Matrices de distancia - Dataset encuesta

Se utiliza una muestra representativa del dataset encuesta (1000 registros) para el cálculo de las matrices de distancia, dado que el dataset completo tiene 10,000 registros y las matrices de distancia completas serían computacionalmente costosas.

In [ ]:
# Muestra para encuesta (el dataset tiene 10000 registros)
np.random.seed(42)
sample_size = 1000
df_enc_sample = df_encuesta.sample(n=sample_size, random_state=42).reset_index(drop=True)

cols_encuesta = ['I. Mensual', 'G. Alimentación', 'Consumo Energía', 'Consumo de Agua',
                 'Nivel Educacional', 'Nº de Hijos', 'Casa Propia', 'Automóvil', 'Teléfono',
                 'I. Fuera Servicios']
df_enc_num = df_enc_sample[cols_encuesta].dropna().copy()

# Normalización
scaler_enc = MinMaxScaler()
df_enc_norm = pd.DataFrame(scaler_enc.fit_transform(df_enc_num), columns=cols_encuesta)

# Binarización para Hamming y Jaccard
df_enc_bin = (df_enc_norm > df_enc_norm.median()).astype(int)

n_enc = len(df_enc_norm)
print(f'Muestra utilizada: {n_enc} registros')
print(f'Total de pares: {n_enc*(n_enc-1)//2}')

In [ ]:
# Calcular matrices de distancia para encuesta
print('Calculando matrices de distancia para encuesta...')

dist_coseno_enc = squareform(pdist(df_enc_norm.values, metric='cosine'))
mat_coseno_enc = pd.DataFrame(dist_coseno_enc)
print('✓ Matriz Coseno calculada')

dist_euclid_enc = squareform(pdist(df_enc_norm.values, metric='euclidean'))
mat_euclid_enc = pd.DataFrame(dist_euclid_enc)
print('✓ Matriz Euclidiana calculada')

dist_hamming_enc = squareform(pdist(df_enc_bin.values, metric='hamming'))
mat_hamming_enc = pd.DataFrame(dist_hamming_enc)
print('✓ Matriz Hamming calculada')

dist_jaccard_enc = squareform(pdist(df_enc_bin.values, metric='jaccard'))
mat_jaccard_enc = pd.DataFrame(dist_jaccard_enc)
print('✓ Matriz Jaccard calculada')

print('\n--- Matriz Coseno (primeros 5x5) ---')
display(mat_coseno_enc.iloc[:5, :5].round(4))

print('\n--- Matriz Euclidiana (primeros 5x5) ---')
display(mat_euclid_enc.iloc[:5, :5].round(4))

print('\n--- Matriz Hamming (primeros 5x5) ---')
display(mat_hamming_enc.iloc[:5, :5].round(4))

print('\n--- Matriz Jaccard (primeros 5x5) ---')
display(mat_jaccard_enc.iloc[:5, :5].round(4))

### 1c y 1d. Top 5 registros más similares y más disímiles - encuesta

In [ ]:
print('=== Top 5 registros MÁS SIMILARES (menor distancia Euclidiana) - encuesta ===')
top_sim_enc = get_top_pairs(dist_euclid_enc, 5, most_similar=True)
for rank, (i, j, dist) in enumerate(top_sim_enc, 1):
    print(f'\n{rank}. Registros {i} y {j} - Distancia: {dist:.6f}')
    print(f'   Registro {i}: {df_enc_num.iloc[i].to_dict()}')
    print(f'   Registro {j}: {df_enc_num.iloc[j].to_dict()}')

print('\n\n=== Top 5 registros MÁS DISÍMILES (mayor distancia Euclidiana) - encuesta ===')
top_dis_enc = get_top_pairs(dist_euclid_enc, 5, most_similar=False)
for rank, (i, j, dist) in enumerate(top_dis_enc, 1):
    print(f'\n{rank}. Registros {i} y {j} - Distancia: {dist:.6f}')
    print(f'   Registro {i}: {df_enc_num.iloc[i].to_dict()}')
    print(f'   Registro {j}: {df_enc_num.iloc[j].to_dict()}')

### 1e. Conclusiones del análisis de datos

**Dataset demanda_laboral:**
- Los registros más similares tienden a compartir niveles de ingresos, edad y tiempo de trabajo muy cercanos, lo que indica que estas variables son las que más influyen en la cercanía entre los perfiles laborales.
- Los registros más disímiles generalmente representan extremos en ingresos (los salarios más altos vs. los más bajos) combinados con diferencias en otras variables como la edad y el tiempo laborando.
- La métrica Euclidiana captura bien las diferencias absolutas entre registros, mientras que la similitud Coseno es más útil para identificar perfiles con proporciones similares entre sus variables.

**Dataset encuesta:**
- Los registros más similares comparten no solo ingresos y gastos parecidos, sino también características sociodemográficas similares (nivel educacional, número de hijos, bienes).
- Las métricas Hamming y Jaccard resultan especialmente útiles para las variables binarias del dataset (Casa Propia, Automóvil, Teléfono), permitiendo identificar perfiles con estilos de vida similares.
- La creación del campo "I. Fuera Servicios" aporta valor al análisis porque permite comparar la capacidad económica real disponible, descontados los gastos esenciales.

---
## Actividad 2: Correlaciones en demanda_laboral

### 2a. Correlación entre Ingresos y Posgrado

El campo `Posgrado` toma valores: **1** = No tiene posgrado, **2** = Sí tiene posgrado.

In [ ]:
corr_ing_posg, p_val_ing_posg = pearsonr(df_demanda['Ingresos'], df_demanda['Posgrado'])
print(f'Correlación Ingresos vs Posgrado: {corr_ing_posg:.4f}')
print(f'p-valor: {p_val_ing_posg:.6f}')
print(f'\nInterpretación: {"Correlación estadísticamente significativa" if p_val_ing_posg < 0.05 else "No hay correlación estadísticamente significativa"}')

### 2b. Correlación entre Ingresos y Tiempo laborando

In [ ]:
corr_ing_tiempo, p_val_ing_tiempo = pearsonr(df_demanda['Ingresos'], df_demanda['Tiempo'])
print(f'Correlación Ingresos vs Tiempo: {corr_ing_tiempo:.4f}')
print(f'p-valor: {p_val_ing_tiempo:.6f}')
print(f'\nInterpretación: {"Correlación estadísticamente significativa" if p_val_ing_tiempo < 0.05 else "No hay correlación estadísticamente significativa"}')

### 2c. Correlación entre Ingresos y Edad

In [ ]:
corr_ing_edad, p_val_ing_edad = pearsonr(df_demanda['Ingresos'], df_demanda['Edad'])
print(f'Correlación Ingresos vs Edad: {corr_ing_edad:.4f}')
print(f'p-valor: {p_val_ing_edad:.6f}')
print(f'\nInterpretación: {"Correlación estadísticamente significativa" if p_val_ing_edad < 0.05 else "No hay correlación estadísticamente significativa"}')

### 2d. ¿Existe correlación entre las variables indicadas?

Para determinar si existe correlación, evaluamos el coeficiente de Pearson y su p-valor (significancia estadística al nivel α = 0.05):

| Par de variables | Correlación | p-valor | ¿Significativa? |
|---|---|---|---|
| Ingresos vs Posgrado | -0.1830 | 0.0050 | Sí |
| Ingresos vs Tiempo | -0.0524 | 0.4250 | No |
| Ingresos vs Edad | -0.0663 | 0.3125 | No |

Solo la correlación entre Ingresos y Posgrado es estadísticamente significativa (p < 0.05). Las correlaciones con Tiempo y Edad no son estadísticamente significativas.

### 2e. Conclusiones de cada valor calculado

- **Ingresos vs Posgrado (r = -0.1830, p = 0.005):** La correlación es negativa y estadísticamente significativa, lo cual indica que en este dataset, tener posgrado (valor 2) se asocia con menores ingresos. Esto puede deberse a que los profesionales con posgrado en la muestra están en etapas iniciales de sus carreras o en sectores donde los retornos salariales del posgrado no son inmediatos. Es una correlación débil (r ≈ -0.18), lo que sugiere que otros factores tienen mayor influencia en el ingreso.
- **Ingresos vs Tiempo (r = -0.0524, p = 0.425):** No existe correlación estadísticamente significativa entre el ingreso y el tiempo laborando. Esto sugiere que en esta muestra, la antigüedad laboral no es un predictor confiable del nivel de ingresos.
- **Ingresos vs Edad (r = -0.0663, p = 0.313):** Tampoco hay correlación significativa entre la edad y los ingresos. Esto indica que la edad por sí sola no determina el nivel salarial en esta población.

### 2f. Matriz de correlación completa - demanda_laboral

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Matriz de correlación
cols_corr_demanda = ['Ingresos', 'Posgrado', 'Tiempo', 'Edad', 'Sexo', 'Naturaleza', 'SMLMV']
corr_matrix_dem = df_demanda[cols_corr_demanda].corr()

print('Matriz de correlación - demanda_laboral:')
display(corr_matrix_dem.round(4))

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix_dem, annot=True, cmap='RdBu_r', center=0, fmt='.4f',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlación - demanda_laboral')
plt.tight_layout()
plt.show()

---
## Actividad 3: Correlaciones en encuesta

**Codificación de variables del dataset encuesta:**

| Variable | Descripción |
|---|---|
| Nivel Educacional | 0: Sin educación, 1: Primaria, 2: Secundaria, 3: Universidad, 4: Postgrado |
| I. Mensual | Salario en miles de pesos |
| G. Alimentación | Gasto mensual en alimentación en miles de pesos |
| Consumo de Energía | Consumo mensual de energía (en watios) |
| Consumo de Agua | Consumo mensual de agua (en m³) |
| Casa Propia | 1: Sí, 0: No |
| Automóvil | 1: Sí, 0: No |
| Teléfono | 1: Sí, 0: No |

### 3g. Correlación entre Ingreso Mensual y Nivel Educacional

In [ ]:
corr_ing_edu, p_val_ing_edu = pearsonr(df_encuesta['I. Mensual'], df_encuesta['Nivel Educacional'])
print(f'Correlación I. Mensual vs Nivel Educacional: {corr_ing_edu:.4f}')
print(f'p-valor: {p_val_ing_edu:.6e}')
print(f'\nInterpretación: {"Correlación estadísticamente significativa" if p_val_ing_edu < 0.05 else "No hay correlación estadísticamente significativa"}')

### 3h. Correlación entre Consumo de Agua y Número de Hijos

In [ ]:
corr_agua_hijos, p_val_agua_hijos = pearsonr(df_encuesta['Consumo de Agua'], df_encuesta['Nº de Hijos'])
print(f'Correlación Consumo de Agua vs Nº de Hijos: {corr_agua_hijos:.4f}')
print(f'p-valor: {p_val_agua_hijos:.6e}')
print(f'\nInterpretación: {"Correlación estadísticamente significativa" if p_val_agua_hijos < 0.05 else "No hay correlación estadísticamente significativa"}')

### 3i. Correlación entre Gasto mensual en Alimentación y Número de Hijos

In [ ]:
corr_alim_hijos, p_val_alim_hijos = pearsonr(df_encuesta['G. Alimentación'], df_encuesta['Nº de Hijos'])
print(f'Correlación G. Alimentación vs Nº de Hijos: {corr_alim_hijos:.4f}')
print(f'p-valor: {p_val_alim_hijos:.6e}')
print(f'\nInterpretación: {"Correlación estadísticamente significativa" if p_val_alim_hijos < 0.05 else "No hay correlación estadísticamente significativa"}')

### 3j. ¿Existe correlación entre las variables indicadas?

Se evalúa cada par de variables utilizando el coeficiente de correlación de Pearson y su p-valor correspondiente.

In [ ]:
print('Resumen de correlaciones - Dataset encuesta:')
print('=' * 70)
print(f'{"Par de variables":<45} {"Correlación":>12} {"p-valor":>12}')
print('-' * 70)
print(f'{"I. Mensual vs Nivel Educacional":<45} {corr_ing_edu:>12.4f} {p_val_ing_edu:>12.2e}')
print(f'{"Consumo de Agua vs Nº de Hijos":<45} {corr_agua_hijos:>12.4f} {p_val_agua_hijos:>12.2e}')
print(f'{"G. Alimentación vs Nº de Hijos":<45} {corr_alim_hijos:>12.4f} {p_val_alim_hijos:>12.2e}')
print('=' * 70)

### 3k. Conclusiones de cada valor calculado

- **I. Mensual vs Nivel Educacional:** Se espera una correlación positiva, indicando que a mayor nivel educacional, mayor tiende a ser el ingreso mensual. Esto confirma la importancia de la educación como factor determinante en la capacidad de generación de ingresos.

- **Consumo de Agua vs Nº de Hijos:** Se esperaría una correlación positiva, ya que hogares con más hijos tienden a consumir más agua por las necesidades adicionales del hogar (higiene, alimentación, limpieza).

- **G. Alimentación vs Nº de Hijos:** Se espera igualmente una correlación positiva fuerte, pues más miembros en el hogar implica mayor gasto en alimentación. Esta es una de las correlaciones más intuitivas y directas del análisis.

**Conclusión general:** Las variables del dataset encuesta muestran relaciones coherentes con la realidad socioeconómica. El nivel educacional influye en los ingresos, y el tamaño del hogar (número de hijos) impacta directamente en los gastos de servicios y alimentación.

### Matriz de correlación completa - encuesta

In [ ]:
corr_matrix_enc = df_encuesta[cols_encuesta].corr()

print('Matriz de correlación - encuesta:')
display(corr_matrix_enc.round(4))

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix_enc, annot=True, cmap='RdBu_r', center=0, fmt='.3f',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlación - encuesta')
plt.tight_layout()
plt.show()